# 02 — Repository implementation walkthrough

This notebook exercises the reusable `pinn` package directly and shows how model, physics, sampling, and trainer components compose into an end-to-end solver.

In [ ]:
import torch
import matplotlib.pyplot as plt
from pinn import MLP, PINNConfig, PINNTrainer, sample_heat_equation, heat_residual, heat_exact_solution
torch.set_default_dtype(torch.float32)
alpha=0.1


## 1. Construct the deterministic training set

The sampler creates separate interior, initial, left-boundary, and right-boundary tensors. A local seed makes repeated calls reproducible without depending on the global random state.

In [ ]:
points=sample_heat_equation(n_interior=2000,n_initial=400,n_boundary=400,seed=7)
for name in ('interior','initial','left_boundary','right_boundary'):
    print(name, getattr(points,name).shape)


## 2. Model and trainer configuration

In [ ]:
model=MLP(input_dim=2,output_dim=1,hidden_dim=64,hidden_layers=4)
config=PINNConfig(alpha=alpha,physics_weight=1.0,initial_weight=20.0,boundary_weight=20.0,learning_rate=1e-3,epochs=1200,log_every=200,seed=7)
trainer=PINNTrainer(model,config)
print(model)


## 3. Inspect the four loss components

The total loss is the weighted sum of physics, initial-condition, and boundary-condition penalties.

In [ ]:
total,physics,initial,boundary=trainer.loss_components(points)
print({"total":float(total),"physics":float(physics),"initial":float(initial),"boundary":float(boundary)})


## 4. Inspect the differentiable PDE residual

In [ ]:
xt=points.interior[:64].clone().requires_grad_(True)
r=heat_residual(model,xt,alpha)
print('residual shape:',tuple(r.shape))
print('requires_grad:',r.requires_grad)
print('RMSE:',torch.sqrt(torch.mean(r.square())).item())


## 5. Train with a callback

In [ ]:
def log(epoch,history):
    print(f'epoch={epoch:4d} total={history.total[-1]:.3e} physics={history.physics[-1]:.3e} ic={history.initial[-1]:.3e} bc={history.boundary[-1]:.3e}')
history=trainer.train(points,callback=log)


In [ ]:
plt.figure(figsize=(8,4))
for k,v in [('total',history.total),('physics',history.physics),('initial',history.initial),('boundary',history.boundary)]: plt.semilogy(v,label=k)
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.show()


## 6. Independent field validation

In [ ]:
x=torch.linspace(-1,1,160); t=torch.linspace(0,1,100)
xx,tt=torch.meshgrid(x,t,indexing='ij')
grid=torch.stack([xx.reshape(-1),tt.reshape(-1)],1)
with torch.no_grad(): pred=trainer.predict(grid)
exact=heat_exact_solution(grid[:,0:1],grid[:,1:2],alpha)
err=pred-exact
print('RMSE:',torch.sqrt(torch.mean(err.square())).item())
print('Linf:',err.abs().max().item())
print('relative L2:',(torch.linalg.vector_norm(err)/torch.linalg.vector_norm(exact)).item())


## 7. Optional L-BFGS refinement

The trainer can perform a quasi-Newton refinement stage after Adam. This is a full-batch optimization stage and should be used with appropriate memory/time constraints.

In [ ]:
refined=PINNTrainer(MLP(hidden_dim=64,hidden_layers=4),PINNConfig(alpha=alpha,initial_weight=20,boundary_weight=20,epochs=500,use_lbfgs=True,lbfgs_steps=40,seed=7))
h2=refined.train(points)
print('Adam-stage final loss:',h2.total[-1])
